# Test DNSE: 1.000 cổ phiếu từ 2016 đến nay

Lấy danh mục từ DNSE, ưu tiên HOSE → HNX → UPCOM, sắp xếp mã trong mỗi sàn rồi chọn tối đa 1.000 mã. Yêu cầu cả 1D, 5m, 1m từ 2016-01-01 đến ngày chạy. Lịch sử thực tế có thể ngắn hơn yêu cầu.

`df` giữ dữ liệu trong bộ nhớ; `coverage` thống kê từng mã/khung; `errors` chứa lỗi đã che thông tin bí mật. Không ghi CSV/Parquet. Notebook chỉ lưu output thống kê và mẫu hiển thị. Lượt chạy có thể mất nhiều phút và dùng nhiều GB RAM. Chỉ phản ánh 1.000 mã đã chọn, không phải toàn bộ thị trường hoặc universe point-in-time.


In [1]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from importlib.metadata import version
import time

import pandas as pd
from IPython.display import display

installed = version("vnstock")
print("Vnstock:", installed)
if int(installed.split(".")[0]) < 6:
    raise RuntimeError("Hãy chạy cell cài đặt vnstock 6, Restart Kernel rồi chạy lại.")

from pathlib import Path
import os
from dotenv import load_dotenv

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "ingest" / "market" / "pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook từ thư mục project.")
load_dotenv(PROJECT_ROOT / ".env", override=True)
missing_keys = [name for name in ("DNSE_API_KEY", "DNSE_API_SECRET")
                if not os.getenv(name, "").strip()]
if missing_keys:
    raise RuntimeError("Thiếu cấu hình trong .env: " + ", ".join(missing_keys))
print("Đã nạp cấu hình DNSE từ .env (không hiển thị key).")

import vnstock
import sys
print("Python kernel:", sys.executable)
print("Module Vnstock:", vnstock.__file__)
print("Vnstock OpenAPI:", version("vnstock-oapi"))


def safe_error(exc):
    message = str(exc)
    for name, value in os.environ.items():
        if any(word in name.upper() for word in ("KEY", "SECRET", "TOKEN", "PASSWORD")) and len(value) > 5:
            message = message.replace(value, "[REDACTED]")
    return message

MAX_SYMBOLS = 1000
WORKERS = 4
REQUEST_GAP = 0.25
SOURCE = "dnse"
INTERVALS = ["1D", "5m", "1m"]
PAUSE_SECONDS = 1

end_date = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
start_date = datetime(2016, 1, 1).date()
print(f"Date range: {start_date} through {end_date}")


Vnstock: 6.0.1
Đã nạp cấu hình DNSE từ .env (không hiển thị key).
Python kernel: /home/mhai/Project DE/A-Lakehouse-Platform-for-Stock-Trading-Strategy-Backtesting-and-Near-Real-Time-Signal-Monitoring/.venv/bin/python
Module Vnstock: /home/mhai/Project DE/A-Lakehouse-Platform-for-Stock-Trading-Strategy-Backtesting-and-Near-Real-Time-Signal-Monitoring/.venv/lib/python3.12/site-packages/vnstock/__init__.py
Vnstock OpenAPI: 0.1.2
Date range: 2016-01-01 through 2026-11-18


In [2]:
catalog = vnstock.Reference(SOURCE).equity.list(type="stock")
listing = catalog.loc[catalog["exchange"].isin(["HOSE", "HNX", "UPCOM"])].copy()
listing["symbol"] = listing["symbol"].astype("string").str.strip().str.upper()
listing = listing.dropna(subset=["symbol"])
listing = listing.loc[listing["symbol"].ne("")]
listing["_exchange_order"] = listing["exchange"].map({"HOSE": 0, "HNX": 1, "UPCOM": 2})
selected = listing.sort_values(["_exchange_order", "symbol"]).drop_duplicates("symbol").head(MAX_SYMBOLS)
SYMBOLS = selected["symbol"].tolist()
print("Catalog distinct symbols:", catalog["symbol"].nunique())
print("Selected:", len(SYMBOLS))
display(selected.groupby("exchange").size().rename("symbols").to_frame())
display(selected[["symbol", "exchange"]].head(10))


Catalog distinct symbols: 1517
Selected: 1000


/home/mhai/Project DE/A-Lakehouse-Platform-for-Stock-Trading-Strategy-Backtesting-and-Near-Real-Time-Signal-Monitoring/.env đọc được bởi người dùng khác trên máy. Nên chạy: chmod 600 /home/mhai/Project DE/A-Lakehouse-Platform-for-Stock-Trading-Strategy-Backtesting-and-Near-Real-Time-Signal-Monitoring/.env
<ipython-input-1-8b080ddf4961>:1: SchemaChangeWarning: SCH-A01 (các route, hồ sơ v5): Cột thế hệ 5 không nguồn chính thức nào gửi thì không có. Xem docs/thay-doi-schema; df.attrs['schema_changes'] liệt kê mọi thay đổi của kết quả.
  catalog = vnstock.Reference(SOURCE).equity.list(type="stock")


,symbols
exchange,
HNX,299
HOSE,410
UPCOM,291


,symbol,exchange
1,AAA,HOSE
3,AAM,HOSE
4,AAN,HOSE
6,AAT,HOSE
11,ABR,HOSE
12,ABS,HOSE
13,ABT,HOSE
15,ACB,HOSE
16,ACC,HOSE
18,ACG,HOSE


In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock, Event

# Limit starts globally across workers; retry a rate limit with a longer wait.
pacing_lock = Lock()
next_start = [0.0]
stop = Event()
def pace():
    with pacing_lock:
        wait = next_start[0] - time.monotonic()
        if wait > 0:
            time.sleep(wait)
        next_start[0] = time.monotonic() + REQUEST_GAP

def fetch_one(symbol, interval):
    record = dict(symbol=symbol, interval=interval, status="FAILED", rows=0,
                  first_time=None, last_time=None, observed_days=0,
                  invalid_ohlc_rows=0, error_type="", message="")
    for attempt in range(3):
        if stop.is_set():
            record["status"] = "STOPPED"
            return record, None
        pace()
        try:
            frame = vnstock.Market(SOURCE).equity(symbol).ohlcv(
                start=start_date.isoformat(), end=end_date.isoformat(),
                interval=interval, compat="v5",
            ).copy()
            record["rows"] = len(frame)
            if frame.empty:
                record["status"] = "EMPTY"
                return record, None
            required = ["time", "open", "high", "low", "close", "volume"]
            absent = [c for c in required if c not in frame]
            if absent:
                raise ValueError("Missing columns: " + ", ".join(absent))
            invalid = (
                frame["high"].lt(frame["low"])
                | frame["open"].lt(frame["low"]) | frame["open"].gt(frame["high"])
                | frame["close"].lt(frame["low"]) | frame["close"].gt(frame["high"])
            )
            times = frame["time"].dropna()
            record.update(status="OK", first_time=str(times.min()), last_time=str(times.max()),
                          observed_days=int(times.dt.date.nunique()), invalid_ohlc_rows=int(invalid.sum()))
            # Metadata is categorical per frame to reduce memory usage.
            frame["symbol"] = pd.Categorical([symbol] * len(frame))
            frame["interval"] = pd.Categorical([interval] * len(frame))
            return record, frame
        except Exception as exc:
            message = safe_error(exc)
            record.update(error_type=type(exc).__name__, message=message)
            if any(code in message.lower() for code in ("429", "rate_limit", "rate limit", "too many")):
                if attempt < 2:
                    time.sleep(30 * (attempt + 1))
                    continue
            if any(code in message.lower() for code in ("401", "403", "unauthorized", "invalid api key")):
                stop.set()
            return record, None

frames = []
records = []
jobs = [(symbol, interval) for symbol in SYMBOLS for interval in INTERVALS]
started = time.monotonic()
print(f"Starting {len(jobs)} requests for {len(SYMBOLS)} symbols", flush=True)
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = [executor.submit(fetch_one, *job) for job in jobs]
    for completed, future in enumerate(as_completed(futures), 1):
        record, frame = future.result()
        records.append(record)
        if frame is not None:
            frames.append(frame)
        if completed % 50 == 0 or completed == len(jobs):
            rows = sum(r["rows"] for r in records if r["status"] == "OK")
            print(f"Progress {completed}/{len(jobs)} | rows={rows:,} | elapsed={time.monotonic()-started:.0f}s", flush=True)
coverage = pd.DataFrame(records).sort_values(["symbol", "interval"]).reset_index(drop=True)
errors = coverage.loc[coverage["status"].ne("OK")].copy()
# Combine only after requests finish. source is constant for this notebook.
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
frames.clear()
if not df.empty:
    df["symbol"] = df["symbol"].astype("category")
    df["interval"] = df["interval"].astype("category")
print(f"Total rows fetched: {len(df):,}")
print(f"Elapsed seconds: {time.monotonic()-started:.1f}")


Starting 3000 requests for 1000 symbols
Progress 50/3000 | rows=103,437 | elapsed=12s
Progress 100/3000 | rows=249,880 | elapsed=25s
Progress 150/3000 | rows=374,768 | elapsed=38s
Progress 200/3000 | rows=463,880 | elapsed=50s
Progress 250/3000 | rows=632,701 | elapsed=63s
Progress 300/3000 | rows=788,378 | elapsed=76s
Progress 350/3000 | rows=895,778 | elapsed=88s
Progress 400/3000 | rows=1,069,937 | elapsed=101s
Progress 450/3000 | rows=1,240,179 | elapsed=114s
Progress 500/3000 | rows=1,425,930 | elapsed=126s
Progress 550/3000 | rows=1,534,085 | elapsed=139s
Progress 600/3000 | rows=1,672,538 | elapsed=152s
Progress 650/3000 | rows=1,816,353 | elapsed=165s
Progress 700/3000 | rows=1,973,989 | elapsed=178s
Progress 750/3000 | rows=2,138,949 | elapsed=190s
Progress 800/3000 | rows=2,265,053 | elapsed=203s
Progress 850/3000 | rows=2,401,083 | elapsed=216s
Progress 900/3000 | rows=2,502,251 | elapsed=228s
Progress 950/3000 | rows=2,616,910 | elapsed=241s
Progress 1000/3000 | rows=2,740,

<ipython-input-1-1d7a520adc4e>:25: DataWarning: VNSTOCK_DATA_EMPTY (market.equity.ohlcv, dnse): nguồn không trả dòng nào. Mã có thể sai, khoảng ngày không có phiên, hoặc nằm ngoài thời gian nguồn lưu dữ liệu.
  frame = vnstock.Market(SOURCE).equity(symbol).ohlcv(
<ipython-input-1-1d7a520adc4e>:25: DataWarning: VNSTOCK_DATA_EMPTY (market.equity.ohlcv, dnse): nguồn không trả dòng nào. Mã có thể sai, khoảng ngày không có phiên, hoặc nằm ngoài thời gian nguồn lưu dữ liệu.
  frame = vnstock.Market(SOURCE).equity(symbol).ohlcv(
<ipython-input-1-1d7a520adc4e>:25: DataWarning: VNSTOCK_DATA_EMPTY (market.equity.ohlcv, dnse): nguồn không trả dòng nào. Mã có thể sai, khoảng ngày không có phiên, hoặc nằm ngoài thời gian nguồn lưu dữ liệu.
  frame = vnstock.Market(SOURCE).equity(symbol).ohlcv(
<ipython-input-1-1d7a520adc4e>:25: DataWarning: VNSTOCK_DATA_EMPTY (market.equity.ohlcv, dnse): nguồn không trả dòng nào. Mã có thể sai, khoảng ngày không có phiên, hoặc nằm ngoài thời gian nguồn lưu dữ liệu.

In [4]:
display(coverage.groupby(["interval", "status"]).agg(
    symbols=("symbol", "nunique"), rows=("rows", "sum")
).reset_index())
ok = coverage.loc[coverage["status"].eq("OK")].copy()
if not ok.empty:
    # Span describes endpoints only, not absence of holes in between.
    ok["first_date"] = pd.to_datetime(ok["first_time"], utc=True).dt.tz_convert("Asia/Ho_Chi_Minh")
    ok["last_date"] = pd.to_datetime(ok["last_time"], utc=True).dt.tz_convert("Asia/Ho_Chi_Minh")
    ok["span_years"] = (ok["last_date"] - ok["first_date"]).dt.total_seconds() / (365.25 * 86400)
    summary = ok.groupby("interval").agg(
        symbols_with_data=("symbol", "nunique"), rows=("rows", "sum"),
        earliest=("first_date", "min"), latest=("last_date", "max"),
        min_span_years=("span_years", "min"), max_span_years=("span_years", "max"),
        invalid_ohlc_rows=("invalid_ohlc_rows", "sum"),
    )
    summary["symbols_span_at_least_10y"] = ok.assign(ten_years=ok["span_years"].ge(10)).groupby("interval")["ten_years"].sum()
    display(summary)
print("Selected symbols:", len(SYMBOLS))
print("Symbols with all requested intervals:", int(coverage.loc[coverage.status.eq("OK")].groupby("symbol")["interval"].nunique().eq(len(INTERVALS)).sum()))
if not df.empty:
    print("DataFrame memory GB:", round(df.memory_usage(deep=True).sum() / 1e9, 3))
    display(df.head(10))
display(errors.head(30))


Selected symbols: 1000
Symbols with all requested intervals: 956
DataFrame memory GB: 0.263


,interval,status,symbols,rows
0,1D,EMPTY,2,0
1,1D,FAILED,2,0
2,1D,OK,996,2040993
3,1m,EMPTY,41,0
4,1m,FAILED,1,0
5,1m,OK,958,2292380
6,5m,EMPTY,41,0
7,5m,OK,959,818077


,symbols_with_data,rows,earliest,latest,min_span_years,max_span_years,invalid_ohlc_rows,symbols_span_at_least_10y
interval,,,,,,,,
1D,996,2040993,2016-01-04 07:00:00+07:00,2026-11-17 07:00:00+07:00,0.021903,10.869268,6091,612
1m,958,2292380,2026-08-18 14:04:00+07:00,2026-11-18 14:13:00+07:00,0.000000,0.251878,0,0
5m,959,818077,2026-08-18 14:05:00+07:00,2026-11-18 14:10:00+07:00,0.000000,0.251863,0,0


,time,open,high,low,close,volume,symbol,interval
0,2016-01-04 07:00:00+07:00,5.83,5.96,5.77,5.91,324250,AAA,1D
1,2016-01-05 07:00:00+07:00,5.87,6.38,5.82,6.33,1842018,AAA,1D
2,2016-01-06 07:00:00+07:00,6.39,6.38,6.09,6.19,249405,AAA,1D
3,2016-01-07 07:00:00+07:00,6.21,6.33,6.06,6.19,334090,AAA,1D
4,2016-01-08 07:00:00+07:00,6.16,6.14,5.96,6.14,168020,AAA,1D
5,2016-01-11 07:00:00+07:00,6.16,6.53,6.09,6.48,614427,AAA,1D
6,2016-01-12 07:00:00+07:00,6.58,6.85,6.38,6.71,916709,AAA,1D
7,2016-01-13 07:00:00+07:00,6.77,6.80,6.53,6.53,189045,AAA,1D
8,2016-01-14 07:00:00+07:00,6.49,6.98,6.43,6.80,802285,AAA,1D
9,2016-01-15 07:00:00+07:00,6.82,7.13,6.76,6.98,509115,AAA,1D


,symbol,interval,status,rows,first_time,last_time,observed_days,invalid_ohlc_rows,error_type,message
160,APL,1m,EMPTY,0,NaN,NaN,0,0,,
161,APL,5m,EMPTY,0,NaN,NaN,0,0,,
172,ARM,1m,EMPTY,0,NaN,NaN,0,0,,
173,ARM,5m,EMPTY,0,NaN,NaN,0,0,,
175,ART,1m,EMPTY,0,NaN,NaN,0,0,,
176,ART,5m,EMPTY,0,NaN,NaN,0,0,,
250,BCV,1m,EMPTY,0,NaN,NaN,0,0,,
251,BCV,5m,EMPTY,0,NaN,NaN,0,0,,
262,BED,1m,EMPTY,0,NaN,NaN,0,0,,
263,BED,5m,EMPTY,0,NaN,NaN,0,0,,


In [5]:
# Per-symbol report stays in memory; inspect any symbol without more API requests.
display(coverage.head(30))
# Example: df.loc[(df["symbol"] == "FPT") & (df["interval"] == "1m")].head()


,symbol,interval,status,rows,first_time,last_time,observed_days,invalid_ohlc_rows,error_type,message
0,A32,1D,OK,1194,2018-10-23 07:00:00+07:00,2026-10-19 07:00:00+07:00,1194,4,,
1,A32,1m,OK,42,2026-08-24 11:07:00+07:00,2026-10-19 11:14:00+07:00,18,0,,
2,A32,5m,OK,33,2026-08-24 11:05:00+07:00,2026-10-19 11:10:00+07:00,18,0,,
3,AAA,1D,OK,2709,2016-01-04 07:00:00+07:00,2026-11-17 07:00:00+07:00,2709,30,,
4,AAA,1m,OK,9772,2026-08-18 14:04:00+07:00,2026-11-18 14:00:00+07:00,64,0,,
5,AAA,5m,OK,2831,2026-08-18 14:05:00+07:00,2026-11-18 13:55:00+07:00,64,0,,
6,AAH,1D,OK,708,2024-01-11 07:00:00+07:00,2026-11-17 07:00:00+07:00,708,0,,
7,AAH,1m,OK,3151,2026-08-18 14:18:00+07:00,2026-11-18 14:02:00+07:00,64,0,,
8,AAH,5m,OK,1863,2026-08-18 14:15:00+07:00,2026-11-18 14:00:00+07:00,64,0,,
9,AAM,1D,OK,2691,2016-01-04 07:00:00+07:00,2026-11-17 07:00:00+07:00,2691,2,,


## Kiểm tra 7 bảng còn lại

Chạy tiếp sau phần OHLCV. Đọc key tùy chọn SSI/FHSC từ `.env`; thiếu key thì báo MISSING_CREDENTIALS hoặc PARTIAL, không sinh dữ liệu thay thế. DNSE secdef chỉ có thông tin giao dịch mã và giá tham chiếu/trần/sàn; chưa phải hồ sơ doanh nghiệp hoặc snapshot quote đầy đủ. Danh mục tích hợp sẵn chỉ là danh mục nhóm chỉ số.

`market_tables` giữ các DataFrame trong RAM, `table_report` mô tả độ phủ, `table_errors` ghi lỗi đã che key. Chỉ snapshot hiện tại, không suy ra lịch sử 10 năm cho bảng tham chiếu.


In [3]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from ingest.market.vnstock.table_probe import run_remaining_tables

market_tables, table_report, table_errors = run_remaining_tables(
    SYMBOLS, start=start_date.isoformat(), end=end_date.isoformat(),
)
display(table_report)
display(table_errors.head(30))


[OK] securities: 1,000 rows (dnse)
instrument_snapshot: 100/1000 requests
instrument_snapshot: 200/1000 requests
instrument_snapshot: 300/1000 requests
instrument_snapshot: 400/1000 requests
instrument_snapshot: 500/1000 requests
instrument_snapshot: 600/1000 requests
instrument_snapshot: 700/1000 requests
instrument_snapshot: 800/1000 requests
instrument_snapshot: 900/1000 requests
instrument_snapshot: 1000/1000 requests
[PARTIAL] securities_details: 6,758 rows (dnse)
[PARTIAL] daily_stock_price: 6,758 rows (dnse)
[PARTIAL] index_list: 9 rows (builtin)
[MISSING_CREDENTIALS] index_components: 0 rows (ssi)
[OK] daily_index: 7,276 rows (dnse)
[OK] intraday_index: 1,791,764 rows (dnse)


,table,source,status,rows,note,failed_requests
0,securities,dnse,OK,1000,Current listing; selected 1000 symbols only.,0
1,securities_details,dnse,PARTIAL,6758,Trading definitions only. Add FHSC_API_KEY for...,0
2,daily_stock_price,dnse,PARTIAL,6758,Current reference/ceiling/floor only; no last ...,0
3,index_list,builtin,PARTIAL,9,"Builtin index groups, not full index catalogue...",0
4,index_components,ssi,MISSING_CREDENTIALS,0,Add SSI_API_KEY and SSI_API_SECRET. No substit...,0
5,daily_index,dnse,OK,7276,4/4 index/interval series returned data; reque...,0
6,intraday_index,dnse,OK,1791764,8/8 index/interval series returned data; reque...,0


,table,source,key,error_type,message
0,instrument_snapshot,dnse,HHG,RateLimitError,Nguồn báo vượt giới hạn tần suất (429) (source...
1,instrument_snapshot,dnse,HHN,RateLimitError,Nguồn báo vượt giới hạn tần suất (429) (source...


In [4]:
for table, frame in market_tables.items():
    print(table, "rows:", len(frame), "columns:", list(frame.columns))
    display(frame.head(3))

index_coverage = []
for table in ("daily_index", "intraday_index"):
    frame = market_tables[table]
    if not frame.empty:
        summary = frame.groupby(["symbol", "interval"]).agg(
            rows=("time", "size"), first_time=("time", "min"), last_time=("time", "max")
        ).reset_index()
        summary["table"] = table
        index_coverage.append(summary)
if index_coverage:
    display(pd.concat(index_coverage, ignore_index=True))


securities rows: 1000 columns: ['symbol', 'exchange', 'type', 'organ_name', 'organ_short_name', 'name_en', 'icb_code', 'icb_name', 'listing_date', 'listed_shares', 'lot_size']
securities_details rows: 6758 columns: ['marketId', 'boardId', 'isin', 'symbol', 'productGrpId', 'securityGroupId', 'basicPrice', 'ceilingPrice', 'floorPrice', 'openInterestQuantity', 'securityStatus', 'symbolAdminStatusCode', 'symbolTradingMethodStatusCode', 'symbolTradingSanctionStatusCode', 'finalTradeDate', 'listingDate', 'time', 'snapshot_at']
daily_stock_price rows: 6758 columns: ['marketId', 'boardId', 'isin', 'symbol', 'productGrpId', 'securityGroupId', 'basicPrice', 'ceilingPrice', 'floorPrice', 'openInterestQuantity', 'securityStatus', 'symbolAdminStatusCode', 'symbolTradingMethodStatusCode', 'symbolTradingSanctionStatusCode', 'finalTradeDate', 'listingDate', 'time', 'snapshot_at']
index_list rows: 9 columns: ['group_name', 'group_code', 'category', 'description']
index_components rows: 0 columns: []
da

,symbol,exchange,type,organ_name,organ_short_name,name_en,icb_code,icb_name,listing_date,listed_shares,lot_size
0,A32,UPCOM,stock,CTCP 32,CTCP 32,<NA>,<NA>,<NA>,2018-10-23 00:00:00+07:00,<NA>,<NA>
1,AAA,HOSE,stock,CTCP Nhựa An Phát Xanh,CTCP Nhựa An Phát Xanh,<NA>,<NA>,<NA>,2016-11-25 00:00:00+07:00,<NA>,<NA>
2,AAH,UPCOM,stock,CTCP Hợp Nhất,CTCP Hợp Nhất,<NA>,<NA>,<NA>,2024-01-11 00:00:00+07:00,<NA>,<NA>


,marketId,boardId,isin,symbol,productGrpId,securityGroupId,basicPrice,ceilingPrice,floorPrice,openInterestQuantity,securityStatus,symbolAdminStatusCode,symbolTradingMethodStatusCode,symbolTradingSanctionStatusCode,finalTradeDate,listingDate,time,snapshot_at
0,STO,G1,VN000000AAA4,AAA,STO,ST,6.83,7.3,6.36,None,NO_HALT,NRM,NRM,NRM,None,2016-11-25T00:00:00Z,2026-11-18 08:00:09.460,2026-11-18T14:29:48.291238+07:00
1,STO,G4,VN000000AAA4,AAA,STO,ST,6.83,7.3,6.36,None,NO_HALT,NRM,NRM,NRM,None,2016-11-25T00:00:00Z,2026-11-18 08:00:09.470,2026-11-18T14:29:48.291238+07:00
2,STO,G7,VN000000AAA4,AAA,STO,ST,6.83,7.3,6.36,None,NO_HALT,NRM,NRM,NRM,None,2016-11-25T00:00:00Z,2026-11-18 08:00:09.480,2026-11-18T14:29:48.291238+07:00


,marketId,boardId,isin,symbol,productGrpId,securityGroupId,basicPrice,ceilingPrice,floorPrice,openInterestQuantity,securityStatus,symbolAdminStatusCode,symbolTradingMethodStatusCode,symbolTradingSanctionStatusCode,finalTradeDate,listingDate,time,snapshot_at
0,STO,G1,VN000000AAA4,AAA,STO,ST,6.83,7.3,6.36,None,NO_HALT,NRM,NRM,NRM,None,2016-11-25T00:00:00Z,2026-11-18 08:00:09.460,2026-11-18T14:29:48.291238+07:00
1,STO,G4,VN000000AAA4,AAA,STO,ST,6.83,7.3,6.36,None,NO_HALT,NRM,NRM,NRM,None,2016-11-25T00:00:00Z,2026-11-18 08:00:09.470,2026-11-18T14:29:48.291238+07:00
2,STO,G7,VN000000AAA4,AAA,STO,ST,6.83,7.3,6.36,None,NO_HALT,NRM,NRM,NRM,None,2016-11-25T00:00:00Z,2026-11-18 08:00:09.480,2026-11-18T14:29:48.291238+07:00


,group_name,group_code,category,description
5,HNX30,HNX30,Chỉ số VN,Chỉ số 30 cổ phiếu hàng đầu HNX
8,VN100,100,Chỉ số VN,100 cổ phiếu có vốn hoá lớn nhất HOSE
9,VN30,30,Chỉ số VN,30 cổ phiếu vốn hóa lớn nhất & thanh khoản tốt...


""


,time,open,high,low,close,volume,symbol,interval,source,asset
0,2016-01-04 07:00:00+07:00,579.03,579.97,573.15,574.41,95930580,VNINDEX,1D,dnse,index
1,2016-01-05 07:00:00+07:00,571.66,575.40,569.26,569.94,97997540,VNINDEX,1D,dnse,index
2,2016-01-06 07:00:00+07:00,569.50,574.57,569.50,574.57,82285160,VNINDEX,1D,dnse,index


,time,open,high,low,close,volume,symbol,interval,source,asset
0,2020-05-11 11:25:00+07:00,823.79,824.52,823.79,824.52,4301792,VNINDEX,5m,dnse,index
1,2020-05-11 11:30:00+07:00,824.42,824.42,824.42,824.42,847392,VNINDEX,5m,dnse,index
2,2020-05-11 13:00:00+07:00,824.47,825.13,824.47,825.13,6640064,VNINDEX,5m,dnse,index


,symbol,interval,rows,first_time,last_time,table
0,HNXINDEX,1D,1599,2020-05-11 07:00:00+07:00,2026-11-17 07:00:00+07:00,daily_index
1,UPCOMINDEX,1D,1398,2020-05-11 07:00:00+07:00,2026-11-17 07:00:00+07:00,daily_index
2,VN30,1D,1598,2020-05-11 07:00:00+07:00,2026-11-17 07:00:00+07:00,daily_index
3,VNINDEX,1D,2681,2016-01-04 07:00:00+07:00,2026-11-17 07:00:00+07:00,daily_index
4,HNXINDEX,1m,386843,2020-05-11 11:28:00+07:00,2026-11-18 14:29:00+07:00,intraday_index
5,HNXINDEX,5m,79617,2020-05-11 11:25:00+07:00,2026-11-18 14:20:00+07:00,intraday_index
6,UPCOMINDEX,1m,376302,2020-05-11 11:28:00+07:00,2026-11-18 14:35:00+07:00,intraday_index
7,UPCOMINDEX,5m,75604,2020-05-11 11:25:00+07:00,2026-11-18 14:30:00+07:00,intraday_index
8,VN30,1m,364619,2020-05-11 11:28:00+07:00,2026-11-18 14:29:00+07:00,intraday_index
9,VN30,5m,74976,2020-05-11 11:25:00+07:00,2026-11-18 14:20:00+07:00,intraday_index


## Snapshot tham chiếu có sẵn: phương án test không cần key mới
Giữ nguyên nguồn và ngày ingest; chưa xác nhận còn hiệu lực. Không dùng làm tham chiếu hiện tại hoặc coi là đủ 1.000 mã. Không ghi file dữ liệu.

Có thể đặt CSV trích từ công bố HOSE/HNX tại `input/market/index_components.csv`. Cột bắt buộc: `index_symbol,symbol,effective_from,source_url`. Mỗi kỳ giữ ngày hiệu lực riêng.


In [2]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from ingest.market.vnstock.table_probe import load_local_reference
local_reference, local_reference_report = load_local_reference(PROJECT_ROOT / "data" / "bronze" / "market")
display(local_reference_report)
for table, frame in local_reference.items():
    print("LOCAL SNAPSHOT:", table)
    display(frame.head(3))

# Có thể nhập CSV đã trích từ công bố của HOSE/HNX vào đường dẫn này.
from ingest.market.vnstock.table_probe import load_exchange_components
exchange_csv = PROJECT_ROOT / "input" / "market" / "index_components.csv"
exchange_components = load_exchange_components(exchange_csv) if exchange_csv.is_file() else pd.DataFrame()
if not exchange_components.empty:
    display(exchange_components.groupby(["index_symbol", "effective_from"]).size().rename("members"))

# Optional test view: keep old index constituents explicitly separate from live results.
if "market_tables" in globals():
    test_tables = dict(market_tables)
    if test_tables["index_components"].empty:
        test_tables["index_components"] = (
            exchange_components if not exchange_components.empty else local_reference["index_components"]
        ).copy()
    if "df" in globals() and not df.empty:
        test_tables["daily_ohlc"] = df.loc[df["interval"].eq("1D")]
        test_tables["intraday_ohlc"] = df.loc[df["interval"].isin(["1m", "5m"])]
    print("Test-view tables:", list(test_tables))
    print("Kiểm tra effective_from hoặc ingested_at trước khi dùng thành phần chỉ số.")


LOCAL SNAPSHOT: index_list
LOCAL SNAPSHOT: index_components
LOCAL SNAPSHOT: securities_details
LOCAL SNAPSHOT: daily_stock_price


,table,rows,symbols,source_snapshot_latest,status,unreadable_files
0,index_list,29,29,2026-10-22T18:17:17.774952+07:00,EXISTING_SNAPSHOT,0
1,index_components,60,30,2026-10-22T18:16:27.177130+07:00,EXISTING_SNAPSHOT,0
2,securities_details,2,1,2026-10-22T18:16:27.787684+07:00,EXISTING_SNAPSHOT,0
3,daily_stock_price,2,1,2026-10-22T18:16:33.735515+07:00,EXISTING_SNAPSHOT,0


,symbol,name,description,full_name,group,index_id,sector_id,source,table,ingested_at,_local_file,_coverage_status
0,VN30,VN30,30 cổ phiếu vốn hóa lớn nhất & thanh khoản tốt...,VN30 Index,HOSE Indices,5.0,NaN,builtin,index_list,2026-10-22T18:17:17.774952+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT
1,VNMID,VNMID,Mid-Cap Index - nhóm cổ phiếu vốn hóa trung bình,VNMidCap Index,HOSE Indices,6.0,NaN,builtin,index_list,2026-10-22T18:17:17.774952+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT
2,VNSML,VNSML,Small-Cap Index - nhóm cổ phiếu vốn hóa nhỏ,VNSmallCap Index,HOSE Indices,7.0,NaN,builtin,index_list,2026-10-22T18:17:17.774952+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT


,symbol,index_symbol,source,table,asset,ingested_at,_local_file,_coverage_status
30,ACB,VN30,vci,index_components,index,2026-10-22T18:16:11.216064+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT
31,BID,VN30,vci,index_components,index,2026-10-22T18:16:11.216064+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT
32,BSR,VN30,vci,index_components,index,2026-10-22T18:16:11.216064+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT


,symbol,exchange,profile,history,num_employees,founded_date,listing_date,charter_capital,website,address,phone,email,tax_id,source,table,asset,ingested_at,_local_file,name,short_name,sector,issued_share,_coverage_status
1,FPT,<NA>,Công ty Cổ phần FPT (FPT) có tiền thân là Công...,<NA>,NaN,NaT,2006-12-13 00:00:00+07:00,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,vci,securities_details,equity,2026-10-22T18:16:11.968455+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,Công ty Cổ phần FPT,FPT Corp,Technology,1.714326e+09,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT
0,FPT,HOSE,- Công nghệ: + Nhóm dịch vụ: Tư vấn Chuyển đổi...,- Ngày 13/09/1988: Tiền thân là Công ty Công n...,54110.0,2002-02-28 00:00:00+07:00,2006-12-13 00:00:00+07:00,18857.0,https://fpt.com,Số 10 - Phố Phạm Văn Bạch - P. Cầu Giấy - Tp. ...,(84.24) 7300 7300,Ir@fpt.com,0101248141,kbs,securities_details,equity,2026-10-22T18:16:27.787684+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,<NA>,<NA>,<NA>,NaN,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT


,symbol,exchange,reference_price,ceiling_price,floor_price,open_price,high_price,low_price,close_price,bid_price_1,bid_vol_1,bid_price_2,bid_vol_2,bid_price_3,bid_vol_3,ask_price_1,ask_vol_1,ask_price_2,ask_vol_2,ask_price_3,ask_vol_3,foreign_buy_volume,foreign_sell_volume,snapshot_date,source,table,asset,ingested_at,_local_file,_coverage_status
1,FPT,HOSE,57400.0,61400.0,53400.0,58000.0,58700.0,57300.0,57400.0,57400.0,70400,57300.0,221600,57200.0,203800,57500.0,7200,57600.0,9000,57700.0,14900,1574890,6161400,2026-10-22,vci,daily_stock_price,equity,2026-10-22T18:16:17.532669+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT
0,FPT,HOSE,57400.0,61400.0,53400.0,58000.0,58700.0,57300.0,57400.0,57400.0,70400,57300.0,221600,57200.0,203800,57500.0,7200,57600.0,9000,57700.0,14900,1574890,6161400,2026-10-22,kbs,daily_stock_price,equity,2026-10-22T18:16:33.735515+07:00,/home/mhai/Project DE/A-Lakehouse-Platform-for...,EXISTING_SNAPSHOT_NOT_VERIFIED_CURRENT
